In [13]:
import time
from pathlib import Path

import requests
import pandas as pd

In [14]:
BASE = "https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities"
START_DATE = "2014-01-01"

TICKERS = [
    "SBER", "GAZP", "LKOH", "GMKN", "NVTK", "ROSN", "TATN", "PLZL", "SNGSP",
    "X5", "MGNT", "CHMF", "NLMK", "ALRS", "AFLT", "IRAO", "RTKM", "MOEX",
    "PHOR", "VTBR", "SIBN", "SMLT", "POSI", "MAGN", "T",
]

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"

In [15]:
def load_history(ticker, start_date=START_DATE):
    """Качает дневную историю по бумаге с MOEX ISS, обходя пагинацию."""
    rows, columns, cursor = [], None, 0
    while True:
        response = requests.get(
            f"{BASE}/{ticker}.json",
            params={"from": start_date, "start": cursor, "iss.meta": "off"},
            timeout=30,
        )
        response.raise_for_status()
        block = response.json()["history"]
        if not block["data"]:
            break
        columns = block["columns"]
        rows.extend(block["data"])
        cursor += len(block["data"])
    df = pd.DataFrame(rows, columns=columns)
    df["TRADEDATE"] = pd.to_datetime(df["TRADEDATE"])
    return df

In [16]:
sber = load_history("SBER")

print(sber.shape)
print(sber["TRADEDATE"].min(), sber["TRADEDATE"].max())
print("пропусков в CLOSE:", sber["CLOSE"].isna().sum())

sber[["TRADEDATE", "SECID", "CLOSE", "VOLUME"]].head()

(3187, 24)
2014-01-06 00:00:00 2026-08-14 00:00:00
пропусков в CLOSE: 18


,TRADEDATE,SECID,CLOSE,VOLUME
0,2014-01-06,SBER,98.91,31691800
1,2014-01-08,SBER,98.19,42372290
2,2014-01-09,SBER,98.00,45986900
3,2014-01-10,SBER,99.20,51902400
4,2014-01-13,SBER,100.25,62051250


### Пропуски в данных

18 дней без цены закрытия у SBER: остановка торгов на МосБирже
(конец февраля — март 2022) и отдельные праздничные дни.
Во всех случаях VOLUME = 0 и NUMTRADES = 0 — торгов не было.

Решение: строки без CLOSE удаляем. Заполнять их последней
известной ценой нельзя — это создало бы несуществующие
наблюдения и исказило распределение доходностей по дням недели.

In [17]:
missing = sber[sber["CLOSE"].isna()]
missing[["TRADEDATE", "CLOSE", "VOLUME", "NUMTRADES"]]

,TRADEDATE,CLOSE,VOLUME,NUMTRADES
2019,2022-01-07,NaN,0,0
2052,2022-02-23,NaN,0,0
2055,2022-02-28,NaN,0,0
2056,2022-03-01,NaN,0,0
2057,2022-03-02,NaN,0,0
2058,2022-03-03,NaN,0,0
2059,2022-03-04,NaN,0,0
2060,2022-03-09,NaN,0,0
2061,2022-03-10,NaN,0,0
2062,2022-03-11,NaN,0,0


In [19]:
sber["CLOSE"].isna().head(10)

0    False
1    False
2    False
3    False
4    False
5    False
6    False
7    False
8    False
9    False
Name: CLOSE, dtype: bool

In [20]:
print(len(sber["CLOSE"]), len(sber["CLOSE"].isna()))

3187 3187


In [21]:
mask = sber["CLOSE"].isna()
sber[mask]

,BOARDID,TRADEDATE,SHORTNAME,SECID,NUMTRADES,VALUE,OPEN,LOW,HIGH,LEGALCLOSEPRICE,...,MARKETPRICE3,ADMITTEDQUOTE,MP2VALTRD,MARKETPRICE3TRADESVALUE,ADMITTEDVALUE,WAVAL,TRADINGSESSION,CURRENCYID,TRENDCLSPR,TRADE_SESSION_DATE
2019,TQBR,2022-01-07,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,293.86,...,291.69,293.86,1.676223e+10,1951406.1,1.676223e+10,0.0,3,SUR,NaN,NaN
2052,TQBR,2022-02-23,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,208.38,...,211.00,208.38,1.295735e+11,567590.0,1.295735e+11,0.0,3,SUR,NaN,NaN
2055,TQBR,2022-02-28,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,4.631605e+10,1663875.0,4.631605e+10,0.0,3,SUR,NaN,NaN
2056,TQBR,2022-03-01,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,4.631605e+10,1663875.0,4.631605e+10,0.0,3,SUR,NaN,NaN
2057,TQBR,2022-03-02,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,1.116498e+11,1663875.0,1.116498e+11,0.0,3,SUR,NaN,NaN
2058,TQBR,2022-03-03,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,4.631605e+10,1663875.0,4.631605e+10,0.0,3,SUR,NaN,NaN
2059,TQBR,2022-03-04,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,4.070000e+11,1663875.0,4.070000e+11,0.0,3,SUR,NaN,NaN
2060,TQBR,2022-03-09,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,2.412234e+11,1663875.0,2.412234e+11,0.0,3,SUR,NaN,NaN
2061,TQBR,2022-03-10,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,1.116498e+11,1663875.0,1.116498e+11,0.0,3,SUR,NaN,NaN
2062,TQBR,2022-03-11,Сбербанк,SBER,0,0.0,NaN,NaN,NaN,130.72,...,130.50,130.72,1.116498e+11,1663875.0,1.116498e+11,0.0,3,SUR,NaN,NaN


In [22]:
r = requests.get(
    f"{BASE}/SBER.json",
    params={"from": "2014-01-01", "start": 0, "iss.meta": "off"},
    timeout=30,
)

print(type(r))
print(r.status_code)
print(r.url)
print(len(r.text), "символов в ответе")

<class 'requests.models.Response'>
200
https://iss.moex.com/iss/history/engines/stock/markets/shares/boards/TQBR/securities/SBER.json?from=2014-01-01&start=0&iss.meta=off
21700 символов в ответе


In [25]:
r.text[:2000]

'{\n"history": {\n\t"columns": ["BOARDID", "TRADEDATE", "SHORTNAME", "SECID", "NUMTRADES", "VALUE", "OPEN", "LOW", "HIGH", "LEGALCLOSEPRICE", "WAPRICE", "CLOSE", "VOLUME", "MARKETPRICE2", "MARKETPRICE3", "ADMITTEDQUOTE", "MP2VALTRD", "MARKETPRICE3TRADESVALUE", "ADMITTEDVALUE", "WAVAL", "TRADINGSESSION", "CURRENCYID", "TRENDCLSPR", "TRADE_SESSION_DATE"], \n\t"data": [\n\t\t["TQBR", "2014-01-06", "Сбербанк", "SBER", 22830, 3154470383.9, 100.2, 98.62, 100.31, 98.63, 99.54, 98.91, 31691800, 99.54, 99.54, 99.54, 3156738969.34, 3156738969.34, 3156738969.34, null, 3, "SUR", -2.23, null],\n\t\t["TQBR", "2014-01-08", "Сбербанк", "SBER", 35633, 4179938515.5, 99.1, 97.85, 99.41, 98.2, 98.65, 98.19, 42372290, 98.65, 98.65, 98.65, 4182984403.36, 4182984403.36, 4182984403.36, null, 3, "SUR", -0.73, null],\n\t\t["TQBR", "2014-01-09", "Сбербанк", "SBER", 41567, 4518388781.2, 98.44, 97.69, 98.77, 97.97, 98.25, 98, 45986900, 98.25, 98.25, 98.25, 4526413769.96, 4526413769.96, 4526413769.96, null, 3, "SUR

In [28]:
data = r.json()
print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['history', 'history.cursor'])


In [30]:
block = data["history"]
print(block.keys())
print(len(block["columns"]))
print(len(block["data"]))

dict_keys(['columns', 'data'])
24
100


In [31]:
print(block["columns"])
print(block["data"][0])

['BOARDID', 'TRADEDATE', 'SHORTNAME', 'SECID', 'NUMTRADES', 'VALUE', 'OPEN', 'LOW', 'HIGH', 'LEGALCLOSEPRICE', 'WAPRICE', 'CLOSE', 'VOLUME', 'MARKETPRICE2', 'MARKETPRICE3', 'ADMITTEDQUOTE', 'MP2VALTRD', 'MARKETPRICE3TRADESVALUE', 'ADMITTEDVALUE', 'WAVAL', 'TRADINGSESSION', 'CURRENCYID', 'TRENDCLSPR', 'TRADE_SESSION_DATE']
['TQBR', '2014-01-06', 'Сбербанк', 'SBER', 22830, 3154470383.9, 100.2, 98.62, 100.31, 98.63, 99.54, 98.91, 31691800, 99.54, 99.54, 99.54, 3156738969.34, 3156738969.34, 3156738969.34, None, 3, 'SUR', -2.23, None]


In [32]:
print(data["history.cursor"])

{'columns': ['INDEX', 'TOTAL', 'PAGESIZE'], 'data': [[0, 3187, 100]]}


In [33]:
frames = []

for i, ticker in enumerate(TICKERS, 1):
    df = load_history(ticker)
    frames.append(df)
    print(f"{i}/{len(TICKERS)} {ticker}: {len(df)} строк")
    time.sleep(0.5)

prices = pd.concat(frames, ignore_index=True)
print(prices.shape)

1/25 SBER: 3187 строк
2/25 GAZP: 3081 строк
3/25 LKOH: 3187 строк
4/25 GMKN: 3081 строк
5/25 NVTK: 3187 строк
6/25 ROSN: 3081 строк
7/25 TATN: 3187 строк
8/25 PLZL: 3081 строк
9/25 SNGSP: 3081 строк
10/25 X5: 407 строк
11/25 MGNT: 3187 строк
12/25 CHMF: 3081 строк
13/25 NLMK: 3081 строк
14/25 ALRS: 3187 строк
15/25 AFLT: 3187 строк
16/25 IRAO: 3177 строк
17/25 RTKM: 3187 строк
18/25 MOEX: 3187 строк
19/25 PHOR: 3187 строк
20/25 VTBR: 3187 строк
21/25 SIBN: 3081 строк
22/25 SMLT: 1471 строк
23/25 POSI: 1182 строк
24/25 MAGN: 3081 строк
25/25 T: 434 строк
(69457, 24)


In [1]:
#frames

In [35]:
prices

,BOARDID,TRADEDATE,SHORTNAME,SECID,NUMTRADES,VALUE,OPEN,LOW,HIGH,LEGALCLOSEPRICE,...,MARKETPRICE3,ADMITTEDQUOTE,MP2VALTRD,MARKETPRICE3TRADESVALUE,ADMITTEDVALUE,WAVAL,TRADINGSESSION,CURRENCYID,TRENDCLSPR,TRADE_SESSION_DATE
0,TQBR,2014-01-06,Сбербанк,SBER,22830,3.154470e+09,100.20,98.62,100.31,98.63,...,99.54,99.54,3.156739e+09,3.156739e+09,3156738969.34,NaN,3,SUR,-2.23,NaN
1,TQBR,2014-01-08,Сбербанк,SBER,35633,4.179939e+09,99.10,97.85,99.41,98.20,...,98.65,98.65,4.182984e+09,4.182984e+09,4182984403.36,NaN,3,SUR,-0.73,NaN
2,TQBR,2014-01-09,Сбербанк,SBER,41567,4.518389e+09,98.44,97.69,98.77,97.97,...,98.25,98.25,4.526414e+09,4.526414e+09,4526413769.96,NaN,3,SUR,-0.19,NaN
3,TQBR,2014-01-10,Сбербанк,SBER,38198,5.109679e+09,97.87,97.52,99.41,99.41,...,98.45,98.45,5.113831e+09,5.113831e+09,5113831362.53,NaN,3,SUR,1.22,NaN
4,TQBR,2014-01-13,Сбербанк,SBER,29942,6.191507e+09,99.30,99.04,100.35,100.24,...,99.78,99.78,6.191738e+09,6.191738e+09,6191738491.4,NaN,3,SUR,1.06,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69452,TQBR,2026-08-10,Т-Техно ао,T,95268,4.835874e+09,277.80,276.46,280.14,277.50,...,277.86,None,2.289061e+09,2.289061e+09,None,0.0,3,SUR,-0.78,2026-08-10
69453,TQBR,2026-08-11,Т-Техно ао,T,83787,4.870157e+09,278.90,278.40,284.16,283.04,...,282.16,None,3.528180e+09,3.528180e+09,None,0.0,3,SUR,1.59,2026-08-11
69454,TQBR,2026-08-12,Т-Техно ао,T,74885,4.374755e+09,283.30,277.02,285.78,280.90,...,282.20,None,2.785216e+09,2.785216e+09,None,0.0,3,SUR,-1.93,2026-08-12
69455,TQBR,2026-08-13,Т-Техно ао,T,130628,7.053150e+09,277.98,264.14,279.16,268.00,...,273.02,None,4.939137e+09,4.939137e+09,None,0.0,3,SUR,-4.69,2026-08-13


In [36]:
summary = prices.groupby("SECID")["TRADEDATE"].agg(["min", "max", "count"])
summary.sort_values("count")

,min,max,count
SECID,,,
X5,2025-01-09,2026-08-14,407
T,2024-11-28,2026-08-14,434
POSI,2021-12-17,2026-08-14,1182
SMLT,2020-10-29,2026-08-14,1471
CHMF,2014-06-09,2026-08-14,3081
GAZP,2014-06-09,2026-08-14,3081
GMKN,2014-06-09,2026-08-14,3081
MAGN,2014-06-09,2026-08-14,3081
SNGSP,2014-06-09,2026-08-14,3081


In [39]:
RAW.mkdir(parents=True, exist_ok=True)

path = RAW / "moex_prices.csv"
prices.to_csv(path, index=False)

#print(path)

print(f"{path.stat().st_size / 1024**2:.1f} МБ")

11.4 МБ


### Состав выборки

- Все 25 бумаг торгуются по последний день выборки, делистинга нет.
- Четыре бумаги с короткой историей: X5 (407 дней), T (434),
  POSI (1182), SMLT (1471) — недавние IPO и смены тикера.
- Девять бумаг начинаются с 2014-06-09: перевод на режим Т+2,
  до этой даты они торговались в другом режиме торгов,
  которого нет в нашей выгрузке.

Решение: оставляем все бумаги. Единица наблюдения —
«бумага в конкретный день», разная длина истории её не искажает.
Следствие: поздние годы представлены большим числом бумаг,
поэтому устойчивость результата проверяем отдельно по подпериодам.